# Uddevalla stock exploration

Run the cells from top to bottom using this project’s Python environment. Install the notebook dependencies and `epsm_data`. Set `UBEM_DATABASE_URL` to a read-only connection to a processed `uddevalla` dataset before starting the kernel. This notebook queries existing evidence; it does not initialize or process a database.


## 1. Connect with the `sweden_ubem` helper

Import `UBEM` and open the `uddevalla` dataset. The helper checks that the dataset exists. Use the explicitly configured connection URL; later cells use this `db` object.


In [ ]:
import os
from sweden_ubem import UBEM

database_url = os.environ["UBEM_DATABASE_URL"]
db = UBEM(database_url=database_url, dataset="uddevalla")
db


The original interactive exploration was checked locally on 14 October 2026. Outputs are omitted from this reusable example; running it requires the declared database and package dependencies.


## 2. Open the building stock and its EPSM catalogue

The database-backed `Stock` keeps the EPSM archetype catalogue with the building collection. It loads building records only when requested. The catalogue exposes the available types, eras, climate zones, renovation states, and ventilation systems.


In [ ]:
stock = db.stock()
catalogue = stock.catalogue
stock


### Matrix categories and their values

A *slot* is one combination of building type, construction era, climate zone, renovation state, and ventilation system. The era ranges come from the catalogue. `SE.06` has no upper year bound. The property list names the values resolved for each slot; it is not an additional classification axis.


In [ ]:
from epsm_data.parametric import PROPERTIES
import pandas as pd

# Display each catalogue axis as a compact, inspectable table.
dimensions = catalogue.dimensions
dimension_table = pd.DataFrame([
    {"axis": "building type", "values": ", ".join(dimensions["categories"])},
    {"axis": "era", "values": ", ".join(
        f"{era} ({start}–{end if end is not None else 'present'})"
        for era, (start, end) in dimensions["eras"].items()
    )},
    {"axis": "climate zone", "values": ", ".join(dimensions["zones"])},
    {"axis": "renovation state", "values": ", ".join(dimensions["states"])},
    {"axis": "ventilation", "values": ", ".join(dimensions["ventilation"])},
    {"axis": "resolved properties", "values": ", ".join(PROPERTIES)},
])
dimension_table


## 3. Uddevalla stock by building type and era

Ask the stock for its type-by-era count matrix. All dataset buildings are counted; missing register years appear in their own column.


In [ ]:
type_era_matrix = stock.matrix()
type_era_matrix


## 4. Focus on multifamily buildings

The same stock matrix can be limited to one EPSM building type.


In [ ]:
mf_type_era_matrix = stock.matrix(building_type="multi_family")
mf_type_era_matrix


## 5. Look inside the SE.01 multifamily cell

A stock cell contains the selected building IDs and their reported ventilation evidence. Counting ventilation within the cell keeps missing or ambiguous EPC evidence explicit. The cell can also be iterated to retrieve full `BuildingRecord` objects.


In [ ]:
mf_se01 = stock.cell(building_type="multi_family", era="SE.01")
mf_se01_ventilation = mf_se01.ventilation_counts()
mf_se01_ventilation


## 6. EPC reported heating for SE.01 multifamily buildings with S ventilation

Select the `S` subset as a stock cell and ask it for EPC heating. The returned table has one row per building and an exclusion reason when a comparable reported value is unavailable. `reported_heating_kwh` is twelve-month delivered heating carrier input, not simulated or ideal thermal demand. Older EPC forms are included only where the parser can separate hot water.


In [ ]:
s_cell = stock.cell(
    building_type="multi_family", era="SE.01", ventilation="S"
)
heating = s_cell.epc_heating()
heating_summary = pd.Series({
    "reported S buildings": len(s_cell),
    "comparable EPC heating": int(heating.reported_heating_kwh.notna().sum()),
    "excluded from heating plot": int(heating.exclusion.notna().sum()),
    "positive reported Atemp": int(heating.reported_heating_kwh_m2.notna().sum()),
}, name="buildings")
heating_summary


### Distribution of reported heating

The left panel shows annual delivered heating carrier input per building. The right panel divides by reported heated area (Atemp) to account for building size. Vertical lines mark medians. Only buildings with a valid value for each measure appear in that panel.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
s_cell.plot_heating(ax=axes[0])
s_cell.plot_heating(per_area=True, ax=axes[1])
fig.suptitle("Uddevalla · multifamily · SE.01 · reported S ventilation")
plt.show()


## 7. Inspect building properties and convert to EPSM

Iterate through a cell to get full `BuildingRecord` objects, or call `frame()` for a compact table of common properties. Each record has `facts`, `components`, `purposes`, `addresses`, `certificates`, and `receipt` for provenance. Conversion is explicit because EPSM needs a weather reference and some records may fail model assembly. `to_epsm` returns successful models **and a receipt for every attempted record**.


In [ ]:
# A compact property table; full BuildingRecord objects remain iterable.
properties = s_cell.frame()
example_record = next(iter(s_cell))
properties.head(5)


### Convert a fully specified catalogue cell when weather is selected

Ventilation is selected from reported evidence. Climate zone and renovation state are catalogue choices, not inferred from the Uddevalla database, so declare them for a model scenario:

```python
model_cell = stock.cell(
    building_type="multi_family", era="SE.01", ventilation="S",
    climate_zone="SE.Zone3", renovation_state="existing",
)
model_cell.archetype_slot  # The exact EPSM Data matrix entry

from sweden_ubem.weather import WeatherReference
weather = WeatherReference.from_epw("/path/to/weather.epw", calendar_year=2023)
models, receipts = model_cell.to_epsm(weather=weather, preflight=False)
```

`models` contains successful EPSM buildings; `receipts` has one entry per attempted building and explains any failed assembly. Supply a per-building weather dictionary or callable when one weather reference is inappropriate for the whole cell. A single `BuildingRecord` also supports `example_record.to_epsm(weather=weather, preflight=False)`.


## 8. Plot reported heated area

Use the same cell directly, or call the convenience plot function on `stock` with filters. Plots return Matplotlib axes for further styling; no manual data extraction or printing is needed.


In [ ]:
stock.plot_atemp(building_type="multi_family", era="SE.01", ventilation="S")
plt.show()


## 9. Height, roof, and storeys in stock properties

`frame()` now includes `storeys` and `basement_storeys` from a uniquely linked, single-building EPC. `height_m` is the representative height from matched geometry evidence. Eave and ridge heights and `roof_type` are from the same height record. Building-level height and roof values are shown only for buildings with **one** footprint component. For multi-component buildings, use `component_frame()` to see each part separately; missing or geometry-mismatched height evidence remains blank.

In [ ]:
# The previously selected SE.01 multifamily S cell is reused here.
physical_columns = ["building_id", "storeys", "basement_storeys", "component_count", "height_m", "eave_height_m", "ridge_height_m", "roof_type"]
physical_properties = s_cell.frame()[physical_columns]
physical_properties.head(10)

In [ ]:
# Component rows preserve height and roof evidence for each part of a building.
components = s_cell.component_frame()
components.head(10)

## 10. What “EPC heating” means

The heating plot uses **reported delivered space-heating carrier input over a measured 12-month period**, not space heating plus domestic hot water (DHW), and not the thermal heat demand of the building.

- **After EPC form v2.6:** sum the separate carrier fields ending in `UPPV` (space heating). DHW is reported separately in fields ending in `VV`; no subtraction is needed for this plotted measure.
- **Before v2.6:** the carrier fields combine space heating and DHW. Subtract `EgiVVBered` (hot-water preparation) to derive space heating. If either amount cannot be resolved, the building is excluded rather than assigned an invented split.

The helper also checks for a unique selected certificate, a 12-month measurement period, a measurement-indicated value, valid energy fields, and one positive carrier family. `epc_heating()` exposes the regime, calculation, older-form operands, and exclusion reason so each plotted value can be traced.

In [ ]:
# Audit how many buildings use each EPC calculation, and inspect the inputs.
heating_audit = s_cell.epc_heating()
heating_basis_counts = (
    heating_audit.assign(reported_heating_basis=heating_audit.reported_heating_basis.fillna("excluded"))
    .groupby(["epc_regime", "reported_heating_basis"], dropna=False)
    .size()
    .rename("buildings")
    .to_frame()
)
heating_basis_counts

In [ ]:
heating_audit[["building_id", "epc_regime", "reported_heating_basis", "combined_heating_dhw_kwh", "dhw_subtracted_kwh", "reported_heating_kwh", "exclusion"]].head(10)